# Setup

In [1]:
!pip install xarray tiler rioxarray

Defaulting to user installation because normal site-packages is not writeable


In [2]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
# from osgeo import gdal
from tqdm import tqdm

%matplotlib inline

/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [3]:
from transformers import AutoImageProcessor
from tiler import Tiler, Merger

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
repo_dir = "lfm"

sys.path.append("lfm")

from lfm.tasks.sem_segmentation.sseg_model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmentation.data_cube_inference import run_datacube_inference, plot_data_cubes

# Config

In [5]:
# Data paths
INPUT_DIR = "/explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/"

# Where to load dinov3 init weights
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/best_model.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs_meeting"  # Change this if you want a specific path
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Training hyperparameters
BATCH_SIZE = 16
NUM_WORKERS = 8

# Number of bands, band filter
NUM_BANDS = 3
BAND_FILTER = [3, 1, 0]

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Output directory: ./outputs_meeting
Using device: cuda


# Load model

In [6]:
# Load model from model factory
encoder = load_dinov3_encoder(weights_local_checkpoint, device=device) 
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    img_size=TARGET_SIZE,
    num_bands=NUM_BANDS,
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint_state, strict=False
)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

Loading model from /explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth


Using cache found in /home/ajkerr1/.cache/torch/hub/facebookresearch_dinov3_main


Encoder loaded with pretrained weights.
Successfully loaded model from checkpoint!


# Load data

## Load statistics from training to normalize inputs

In [7]:
print("\n" + "="*60)
print("STEP 1: Loading training dataset statistics.")
print("="*60)

# Load mean and std from a previous training run
MEAN = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_mean.npy")
MEAN = MEAN[BAND_FILTER]
STD = np.load("/explore/nobackup/projects/lfm/model_inference/stats/sem_seg/dataset_std.npy")
STD = STD[BAND_FILTER]
print(MEAN, STD)


STEP 1: Loading training dataset statistics.
[0.35927844 0.35267626 0.3384354 ] [0.12096693 0.12117315 0.11590558]


# Inference

In [8]:
# plot some data cubes
plot_data_cubes(INPUT_DIR, 5, 25, ".")

Loading 5 TIFF files...
Opened Cube-LTM30N_Zoom-5_Tile-1-22.tif with 25 bands

MASK CHANGE DETECTION
Unmasked band min, max: -3.4028227e+38 -3.4028227e+38
Number of masked values: 0
Range of unmasked values: [-3.4028226550889045e+38, -3.4028226550889045e+38]
Number of unique unmasked values: 1
Unmasked band min, max: -3.4028227e+38 -3.4028227e+38
Number of masked values: 0
Range of unmasked values: [-3.4028226550889045e+38, -3.4028226550889045e+38]
Number of unique unmasked values: 1
Unmasked band min, max: -3.4028227e+38 -3.4028227e+38
Number of masked values: 0
Range of unmasked values: [-3.4028226550889045e+38, -3.4028226550889045e+38]
Number of unique unmasked values: 1
Unmasked band min, max: -3.4028227e+38 -3.4028227e+38
Number of masked values: 0
Range of unmasked values: [-3.4028226550889045e+38, -3.4028226550889045e+38]
Number of unique unmasked values: 1
Unmasked band min, max: -3.4028227e+38 -3.4028227e+38
Number of masked values: 0
Range of unmasked values: [-3.402822655088

In [9]:
# Run inference, create viz of inference
n_images = 3
images, predictions = run_datacube_inference(
    model=model,
    device=device,
    tiff_dir=INPUT_DIR,
    mean=MEAN,
    std=STD,
    output_path="sem_seg_inf_test_3_19.png",
    n_images=n_images,
    model_native_size=TARGET_SIZE[0],
    tile_overlap=0.25,
    threshold=0.75,
    normalize=True,
    save_inputs_dir=None,
    use_sliding=True,
)

Found 1679 TIFF files
Loading 3 images from TIFF files...
Extracting images with mean=[0.35927844 0.35267626 0.3384354 ], std=[0.12096693 0.12117315 0.11590558]
Processing /explore/nobackup/people/ajkerr1/Lunar_FM/sandy_cubes/data_cubes/5_band_WAC/Cube-LTM30N_Zoom-5_Tile-1-22.tif: 564 bands, 112 complete 5-band slices
Image shape before norm, after resize: (512, 512, 3)
Mean, std shapes before norm: ((1, 1, 3), (1, 1, 3))
  ✓ Slice 0: bands 0-2 → valid
Image shape before norm, after resize: (512, 512, 3)
Mean, std shapes before norm: ((1, 1, 3), (1, 1, 3))
  ✓ Slice 1: bands 5-7 → valid
Image shape before norm, after resize: (512, 512, 3)
Mean, std shapes before norm: ((1, 1, 3), (1, 1, 3))
  ✓ Slice 2: bands 10-12 → valid

Final extracted array shape: (3, 512, 512, 3)
Image 0 shape: (512, 512, 3), min: -2935857479279359686540018650994154930176.00, max: -2808231658030753234892033037502626398208.00
Image 1 shape: (512, 512, 3), min: -2935857479279359686540018650994154930176.00, max: -28

100%|██████████| 4/4 [00:00<00:00,  5.64 batches/s]


Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 0.00]

Processing image 2/3
Number of tiles: 4


100%|██████████| 4/4 [00:00<00:00, 43.14 batches/s]


Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 0.00]

Processing image 3/3
Number of tiles: 4


100%|██████████| 4/4 [00:00<00:00, 42.93 batches/s]


Merged probabilities shape: (512, 512, 1)
Final prediction shape: (512, 512)
Prediction range: [0.00, 0.00]

Got 3 predictions
Output mask shapes: [(512, 512), (512, 512), (512, 512)]

Creating visualization...

Saved visualization to: sem_seg_inf_test_3_19.png
